In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, chi2_contingency

pd.set_option('display.max_columns', None)
plt.style.use('ggplot')

In [ ]:
# 1) Load data and understand structure
df = pd.read_csv('hr_attribution.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
# check datatypes
print('Data types:')
print(df.dtypes)

print('First 5 rows:')
display(df.head())


In [ ]:
# find missing values
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

In [ ]:
print('Attrition distribution:')
print(df['attrition'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

In [ ]:
df['education_field'].unique()

In [ ]:
df['department'].unique()

In [ ]:
df['employment_type'].unique()

In [ ]:
# 2) Data cleaning with targeted imputations
clean_df = df.copy()

# Drop duplicate employee records if any
before = clean_df.shape[0]
clean_df = clean_df.drop_duplicates(subset=['employee_id'])
after = clean_df.shape[0]
print(f'Dropped duplicate employee records: {before - after}')

In [ ]:

# Standardize string values
for col in clean_df.select_dtypes(include='object').columns:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# Missing value overview before imputation
print('Missing values before imputation:')
print(clean_df[['monthly_income', 'job_satisfaction', 'performance_rating']].isnull().sum())

num_cols = clean_df.select_dtypes(include='number').columns.tolist()
cat_cols = clean_df.select_dtypes(exclude='number').columns.tolist()

In [ ]:
# MONTHLY INCOME IMPUTATION (similar bucket: job_role + job_level)
if 'monthly_income' in num_cols and clean_df['monthly_income'].isnull().any():
    income_group_med = clean_df.groupby(['job_role', 'job_level'])['monthly_income'].median()
    null_idx = clean_df[clean_df['monthly_income'].isnull()].index
    for idx in null_idx:
        key = (clean_df.loc[idx, 'job_role'], clean_df.loc[idx, 'job_level'])
        if key in income_group_med.index:
            clean_df.loc[idx, 'monthly_income'] = income_group_med.loc[key]
    if clean_df['monthly_income'].isnull().any():
        clean_df['monthly_income'] = clean_df['monthly_income'].fillna(clean_df['monthly_income'].median())
        


In [ ]:
# JOB SATISFACTION AND PERFORMANCE RATING IMPUTATION
for target_col in ['job_satisfaction', 'performance_rating']:
    if target_col in clean_df.columns and clean_df[target_col].isnull().any():
        target_mode = clean_df[target_col].mode(dropna=True)
        if not target_mode.empty:
            clean_df[target_col] = clean_df[target_col].fillna(target_mode.iloc[0])
        else:
            clean_df[target_col] = clean_df[target_col].fillna(clean_df[target_col].median())

print('Job satisfaction missing values after mode imputation:', clean_df['job_satisfaction'].isnull().sum())
print('Performance rating missing values after mode imputation:', clean_df['performance_rating'].isnull().sum())

In [ ]:
# Verify group-based imputation for monthly_income
print('Monthly Income Imputation by Job Role & Job Level ')
print('Median monthly income by job_role and job_level:')
income_medians = clean_df.groupby(['job_role', 'job_level'])['monthly_income'].agg(['median', 'count', 'mean']).round(2)
display(income_medians)

print(f'Monthly income missing values after group imputation: {clean_df["monthly_income"].isnull().sum()}')
print(f'Remaining missing values by column:')
display(clean_df.isnull().sum()[clean_df.isnull().sum() > 0].sort_values(ascending=False))

In [ ]:
# Diagnostic: Check which job_role/job_level combinations have ALL null monthly_income
print('Groups with NULL monthly_income values')
null_mask = clean_df['monthly_income'].isnull()
if null_mask.any():
    null_by_group = clean_df[null_mask].groupby(['job_role', 'job_level']).size()
    print(f'Groups with null values: {len(null_by_group)}')
    display(null_by_group)
    
    # Check if these groups have non-null values elsewhere
    print('\nNon-null counts by these groups:')
    for (job_role, job_level) in null_by_group.index:
        group_data = clean_df[(clean_df['job_role'] == job_role) & (clean_df['job_level'] == job_level)]
        non_null_count = group_data['monthly_income'].notna().sum()
        total_count = len(group_data)
        print(f'  {job_role} (Level {job_level}): {non_null_count} non-null / {total_count} total')

In [ ]:
# 3) Exploratory Data Analysis (EDA)

# Attrition count
plt.figure(figsize=(8, 5))
attrition_counts = clean_df['attrition'].value_counts().reindex(['No', 'Yes']).fillna(0)
plt.bar(attrition_counts.index, attrition_counts.values, color=['#4C78A8', '#F58518'])
plt.title('Attrition Class Balance')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Monthly income by attrition
plt.figure(figsize=(8, 5))
income_no = clean_df.loc[clean_df['attrition'] == 'No', 'monthly_income'].dropna()
income_yes = clean_df.loc[clean_df['attrition'] == 'Yes', 'monthly_income'].dropna()
plt.boxplot(
    [income_no, income_yes],
    labels=['No', 'Yes'],
    patch_artist=True,
    boxprops=dict(facecolor='#72B7B2', color='#2F4B7C'),
    medianprops=dict(color='#E45756', linewidth=2),
    whiskerprops=dict(color='#2F4B7C'),
    capprops=dict(color='#2F4B7C')
)
plt.title('Monthly Income vs Attrition')
plt.xlabel('Attrition')
plt.ylabel('Monthly Income')
plt.tight_layout()
plt.show()

# Overtime hours by attrition
plt.figure(figsize=(8, 5))
overtime_no = clean_df.loc[clean_df['attrition'] == 'No', 'overtime_hours_per_week'].dropna()
overtime_yes = clean_df.loc[clean_df['attrition'] == 'Yes', 'overtime_hours_per_week'].dropna()
plt.boxplot(
    [overtime_no, overtime_yes],
    labels=['No', 'Yes'],
    patch_artist=True,
    boxprops=dict(facecolor='#54A24B', color='#2F4B7C'),
    medianprops=dict(color='#E45756', linewidth=2),
    whiskerprops=dict(color='#2F4B7C'),
    capprops=dict(color='#2F4B7C')
)
plt.title('Overtime Hours vs Attrition')
plt.xlabel('Attrition')
plt.ylabel('Overtime Hours per Week')
plt.tight_layout()
plt.show()

# Work-life balance by attrition
plt.figure(figsize=(9, 5))
wlb_table = pd.crosstab(clean_df['work_life_balance'], clean_df['attrition']).reindex(sorted(clean_df['work_life_balance'].dropna().unique()))
wlb_table.plot(kind='bar', color=['#4C78A8', '#F58518'], width=0.8)
plt.title('Work-life Balance by Attrition')
plt.xlabel('Work-life Balance')
plt.ylabel('Count')
plt.legend(title='Attrition')
plt.tight_layout()
plt.show()

# Correlation heatmap for all numeric relations
numeric_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
if 'employee_id' in numeric_cols:
    numeric_cols.remove('employee_id')

corr = clean_df[numeric_cols].corr()
plt.figure(figsize=(14, 11))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha='right')
plt.yticks(range(len(numeric_cols)), numeric_cols)

for row_index in range(len(numeric_cols)):
    for col_index in range(len(numeric_cols)):
        value = corr.iloc[row_index, col_index]
        box_style = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='gray', alpha=0.85)
        plt.text(col_index, row_index, f'{value:.2f}', ha='center', va='center', fontsize=6, bbox=box_style)

plt.colorbar(label='Correlation')
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# 4) Feature engineering
feat_df = clean_df.copy()

# Ratio features
feat_df['tenure_ratio'] = feat_df['years_in_current_role'] / (feat_df['years_at_company'] + 1)
feat_df['income_per_experience_year'] = feat_df['monthly_income'] / (feat_df['total_working_years'] + 1)

# Binary behavior flags
feat_df['is_frequent_traveler'] = feat_df['business_travel'].isin(['Travel_Frequently']).astype(int)
feat_df['is_remote_heavy'] = (feat_df['remote_work_days_per_week'] >= 3).astype(int)

# Bucketized distance feature
feat_df['distance_bucket'] = pd.cut(
    feat_df['distance_from_home_km'],
    bins=[-1, 5, 15, 30, np.inf],
    labels=['Very_Near', 'Near', 'Moderate', 'Far']
)

print('Feature engineered dataset shape:', feat_df.shape)
display(feat_df[['employee_id', 'tenure_ratio', 'income_per_experience_year', 'is_frequent_traveler', 'is_remote_heavy', 'distance_bucket']].head())

In [ ]:
# 5) Hypothesis testing
alpha = 0.05

# Hypothesis 1: Business travel is associated with attrition.
# H0: Attrition is independent of business travel frequency.
# H1: Attrition depends on business travel frequency.

travel_table = pd.crosstab(feat_df['business_travel'], feat_df['attrition'])
chi2_stat, p_value_travel, dof, expected = chi2_contingency(travel_table)

print('Chi-Square Test (Business Travel vs Attrition):')
print('Contingency Table:')
display(travel_table)
print(f'  chi2 = {chi2_stat:.4f}, dof = {dof}, p-value = {p_value_travel:.3e}')
print('  Decision:', 'Reject H0' if p_value_travel < alpha else 'Fail to reject H0')
print()

# Hypothesis 2: Job satisfaction differs between attrition groups.
# H0: Mean job satisfaction is equal for attrition Yes and No groups.
# H1: Mean job satisfaction differs between attrition Yes and No groups.

sat_yes = feat_df.loc[feat_df['attrition'] == 'Yes', 'job_satisfaction'].dropna()
sat_no = feat_df.loc[feat_df['attrition'] == 'No', 'job_satisfaction'].dropna()

t_stat_sat, p_value_sat = ttest_ind(sat_yes, sat_no, equal_var=False)

print('T-Test (Job Satisfaction by Attrition):')
print(f'  Mean satisfaction (Attrition=Yes): {sat_yes.mean():.2f}')
print(f'  Mean satisfaction (Attrition=No):  {sat_no.mean():.2f}')
print(f'  t-statistic = {t_stat_sat:.4f}, p-value = {p_value_sat:.3e}')
print('  Decision:', 'Reject H0' if p_value_sat < alpha else 'Fail to reject H0')
print()

# Hypothesis 3: Overtime is associated with attrition.
# H0: Mean overtime hours are equal for attrition Yes and No groups.
# H1: Mean overtime hours differ between attrition Yes and No groups

overtime_yes = feat_df.loc[feat_df['attrition'] == 'Yes', 'overtime_hours_per_week'].dropna()
overtime_no = feat_df.loc[feat_df['attrition'] == 'No', 'overtime_hours_per_week'].dropna()

t_stat_ot, p_value_ot = ttest_ind(overtime_yes, overtime_no, equal_var=False)

print('T-Test (Overtime Hours by Attrition):')
print(f'  Mean overtime (Attrition=Yes): {overtime_yes.mean():.2f}')
print(f'  Mean overtime (Attrition=No):  {overtime_no.mean():.2f}')
print(f'  t-statistic = {t_stat_ot:.4f}, p-value = {p_value_ot:.3e}')
print('  Decision:', 'Reject H0' if p_value_ot < alpha else 'Fail to reject H0')